# Library System
A simple console library with books, members, borrowing, and returning. Premium members can borrow more books.

In [ ]:
class Book:
    def __init__(self, book_id, title, author):
        self.book_id, self.title, self.author = book_id, title, author
        self._is_available = True

    def borrow(self):
        if not self._is_available: return False
        self._is_available = False
        return True

    def return_book(self): self._is_available = True
    def is_available(self): return self._is_available

    def __str__(self):
        status = "Available" if self._is_available else "Borrowed"
        return f"{self.book_id}: {self.title} - {self.author} ({status})"


class Member:
    def __init__(self, member_id, name):
        self.member_id, self.name = member_id, name
        self._borrowed_books = []

    def get_borrow_limit(self): return 2

    def borrow_book(self, book):
        if len(self._borrowed_books) == self.get_borrow_limit(): return False
        if not book.borrow(): return False
        self._borrowed_books.append(book)
        return True

    def return_book(self, book):
        if book not in self._borrowed_books: return False
        self._borrowed_books.remove(book)
        book.return_book()
        return True

    def list_borrowed_books(self): return self._borrowed_books
    def __str__(self): return f"{self.name} ({self.member_id})"


class PremiumMember(Member):
    def get_borrow_limit(self): return 4


class Library:
    def __init__(self, name):
        self.name, self.books, self.members = name, {}, {}

    def add_book(self, book):
        if book.book_id in self.books: raise ValueError("Book ID already used.")
        self.books[book.book_id] = book

    def register_member(self, member):
        if member.member_id in self.members: raise ValueError("Member ID already used.")
        self.members[member.member_id] = member

    def find_book(self, query):
        q = query.lower()
        return [b for b in self.books.values() if q in b.title.lower() or q in b.author.lower()]

    def display_available_books(self):
        return [b for b in self.books.values() if b.is_available()]

    def lend_book(self, member_id, book_id):
        member, book = self.members.get(member_id), self.books.get(book_id)
        if member is None or book is None: return False
        return member.borrow_book(book)

    def receive_return(self, member_id, book_id):
        member, book = self.members.get(member_id), self.books.get(book_id)
        return member is not None and book is not None and member.return_book(book)


def make_library():
    lib = Library("My Library")
    for b in [Book("1", "The Hobbit", "Tolkien"),
              Book("2", "Matilda", "Dahl"), Book("3", "Clean Code", "Martin")]:
        lib.add_book(b)
    lib.register_member(Member("1", "Sara"))
    lib.register_member(PremiumMember("2", "Omar"))
    return lib


def main(library=None, ask=input, say=print):
    if library is None: library = make_library()
    while True:
        say("\n1 Books  2 Search  3 Add book  4 Register member")
        say("5 Borrow  6 Return  7 Member books  8 Exit")
        try:
            choice = int(ask("Choose 1-8: "))
            if choice == 1:
                for b in library.display_available_books(): say(str(b))
            elif choice == 2:
                for b in library.find_book(ask("Search title/author: ")): say(str(b))
            elif choice == 3:
                library.add_book(Book(ask("ID: "), ask("Title: "), ask("Author: ")))
                say("Book added.")
            elif choice == 4:
                mid, name = ask("Member ID: "), ask("Name: ")
                kind = ask("1 Regular, 2 Premium: ")
                cls = {"1": Member, "2": PremiumMember}.get(kind)
                if cls is None: raise ValueError("Choose 1 or 2.")
                library.register_member(cls(mid, name))
                say("Member registered.")
            elif choice in (5, 6):
                mid, bid = ask("Member ID: "), ask("Book ID: ")
                ok = library.lend_book(mid, bid) if choice == 5 else library.receive_return(mid, bid)
                say("Done." if ok else "Could not complete that action.")
            elif choice == 7:
                member = library.members.get(ask("Member ID: "))
                if member is None: say("Member not found.")
                else:
                    say(str(member))
                    for b in member.list_borrowed_books(): say(str(b))
            elif choice == 8:
                say("Goodbye!")
                return library
            else: say("Choose a number from 1 to 8.")
        except ValueError as error:
            say(f"Invalid input: {error}")



## Start the menu
Choose from options 1-8.

In [ ]:
library = main()


## Quick check
This checks the menu, unavailable books, and the borrowing limit.

In [ ]:
# Try every menu option once, then check unavailable books and the borrowing limit.
answers = iter(["1", "2", "Hobbit", "3", "4", "Dune", "Herbert",
                "4", "3", "Ali", "1", "5", "1", "1", "6", "1", "1",
                "7", "1", "8"])
output = []
lib = main(make_library(), ask=lambda prompt: next(answers), say=output.append)
assert "Book added." in output and "Member registered." in output and "Goodbye!" in output
assert "4" in lib.books and "3" in lib.members
lib = make_library()
assert lib.lend_book("1", "1")
assert not lib.lend_book("2", "1")  # already borrowed
assert lib.lend_book("1", "2")
assert not lib.lend_book("1", "3")  # regular member limit reached
print("All quick checks passed.")
